In [1]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableLambda
import warnings

In [2]:
# Suppress all UserWarnings coming specifically from the pydantic module and its submodules

warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    module="pydantic.*"
)

## Logical Routing

In [3]:
# Define the Data Model 

class RouteQuery(BaseModel):
    """Route a user query to the most relevant datasource."""
    datasource: Literal["python_docs", "js_docs", "golang_docs"] = Field(
        ...,
        description="Given a user question, choose which datasource is most relevant.",
    )

In [4]:
# LLM with function call 

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
structured_llm = llm.with_structured_output(RouteQuery)

In [5]:
# Create the Prompt

system_instruction = """You are an expert at routing a user question to the appropriate data source.
Based on the programming language the question is referring to, route it to the relevant data source."""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_instruction),
    ("human", "{question}"),
])

In [6]:
# Build the First Half: The Router Chain. This evaluates the question and outputs a RouteQuery object.
router = prompt | structured_llm

In [7]:
# Define the Execution Logic (The Switchboard)

def choose_route(routing_decision: RouteQuery):
    """Takes the LLM's decision and triggers the actual downstream chains."""
    
    # In a real app, you would return actual LangChain retrievers/chains here
    if routing_decision.datasource == "python_docs":
        return "Route chosen: Python. --> Executing Python RAG Chain..."
        
    elif routing_decision.datasource == "js_docs":
        return "Route chosen: JavaScript. --> Executing JS RAG Chain..."
        
    elif routing_decision.datasource == "golang_docs":
        return "Route chosen: Golang. --> Executing Go RAG Chain..."
        
    else:
        return "Error: Unknown datasource."

In [ ]:
# Assemble the Full Pipeline, Connect the router's output directly into the execution logic

full_chain = router | RunnableLambda(choose_route)

In [11]:
# Test the Pipeline
test_question = """Why doesn't the following code work:

from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages(["human", "speak in {language}"])
prompt.invoke("french")
"""

result = router.invoke({"question": test_question})

In [13]:
result

RouteQuery(datasource='python_docs')

In [14]:
type(result)

__main__.RouteQuery

In [15]:
result.datasource

'python_docs'

In [16]:
print(full_chain.invoke({"question": test_question}))

Route chosen: Python. --> Executing Python RAG Chain...


## Semantic Routing